# 03. Blind hand-labelling sheet and the agreement with the rater

**Design decisions.**
- One Excel sheet, `Hiring_Research/labelling/hand_labels_sheet.xlsx`, with two blocks: `LinkedIn2024`, a stratified sample of the rebuilt 2024 pool (N_PER_LABEL rated 1 and N_PER_LABEL rated 0 by the new rater, shuffled), and `Monster2019`, every 2019 advertisement in the subset. The sheet shows the title and the same opening the rater read, and no rater output, so the labelling is blind. A second tab holds the six-part rubric.
- The mapping from sheet row to advertisement id is kept next to the sheet (`sheet_key.csv`), not in the sheet.
- Stage A writes the sheet once and never overwrites it. Stage B reads it back; if every row of a block is labelled, it computes agreement, Cohen's kappa, precision and recall of the rater against the human labels for that block and writes `reports/validation_<block>.json` and a text-free `reports/validation_labels.csv`. Unfilled blocks are reported and skipped.


In [ ]:
# === SETTINGS ===
import os
from pathlib import Path
REPO_URL   = "https://github.com/anasbiswas1/hiring-research.git"
DRIVE_ROOT = Path(os.environ.get("HIRING_DRIVE_ROOT", "/content/drive/MyDrive/Hiring_Research"))
REPO_DIR   = DRIVE_ROOT / "hiring-research"
MODEL         = "claude-sonnet-5-5"   # the rater whose output is validated (same as notebook 02)
N_PER_LABEL   = 20                    # 2024 block: this many rated 1 and this many rated 0
OPENING_CHARS = 700
SEED          = 7
LABELLING_DIR = DRIVE_ROOT / "labelling"


In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import os, sys, shutil, subprocess, importlib
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
CRED = DRIVE_ROOT / ".secrets"
for name, dot in [("git-credentials", ".git-credentials"), ("gitconfig", ".gitconfig")]:
    if (CRED / name).exists():
        shutil.copy(CRED / name, Path.home() / dot)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"])
os.environ["HIRING_DRIVE_ROOT"] = str(DRIVE_ROOT)
os.environ["HIRING_REPO_ROOT"] = str(REPO_DIR)
assert (REPO_DIR / ".git").exists(), f"no git checkout at {REPO_DIR}: run notebook 00 Stage 0 first"
os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()
pull = subprocess.run(["git", "pull", "origin", "main"], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())
print("repo:", REPO_DIR)


## Stage 0. Install

In [ ]:
# === INSTALL ===
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *['openpyxl', 'pandas']], capture_output=True, text=True)
print("pip -> rc", r.returncode)
print(r.stderr[-600:])


## Stage A. Write the blind labelling sheet (once)

In [ ]:
# === STAGE A: build the sheet once ===
import importlib
import pandas as pd
import ads, validation, rater
importlib.reload(validation)
SHEET = LABELLING_DIR / "hand_labels_sheet.xlsx"
KEY = LABELLING_DIR / "sheet_key.csv"
ratings_file = REPO_DIR / "reports" / f"ratings_{MODEL}.csv"
assert ratings_file.exists(), f"{ratings_file} not found: run notebook 02 first"
FIN = pd.read_csv(ratings_file, dtype={"ad_id": str})
SUB = REPO_DIR / "data" / "subsets"
ADS = pd.concat([pd.read_csv(p, dtype={"ad_id": str}) for p in sorted(SUB.glob("*.csv"))]).drop_duplicates("ad_id")
if SHEET.exists():
    print("sheet already exists, not overwritten:", SHEET)
else:
    pool24 = FIN[(FIN.year == 2024) & FIN.in_pool].dropna(subset=["final"])
    block24 = validation.stratified_sample(pool24, N_PER_LABEL, SEED)
    block19 = FIN[(FIN.year == 2019) & FIN.in_pool]
    sheet, key = validation.build_sheet({"LinkedIn2024": block24, "Monster2019": block19}, ADS, OPENING_CHARS, SEED)
    validation.write_sheet_xlsx(sheet, SHEET)
    key.to_csv(KEY, index=False)
    print(f"sheet written: {SHEET}  ({len(block24)} LinkedIn 2024 + {len(block19)} Monster 2019 rows)")
    print("fill the column human_senior with 1 or 0 for every row, save, then run Stage B")


## Stage B. Agreement with the rater (run after the sheet is filled in)

In [ ]:
# === STAGE B: agreement per block, once the sheet is filled ===
import json
import pandas as pd
import ads, validation
sheet = validation.read_sheet(SHEET)
key = pd.read_csv(KEY, dtype=str)
FIN = pd.read_csv(REPO_DIR / "reports" / f"ratings_{MODEL}.csv", dtype={"ad_id": str})
m = sheet.merge(key, on="sheet_id").merge(FIN[["ad_id", "final"]], on="ad_id", how="left")
rows = []
for block, g in m.groupby("source"):
    filled = g.human_senior.notna() & g.final.notna()
    if not filled.all():
        print(f"{block}: {int((~g.human_senior.notna()).sum())} of {len(g)} rows still unlabelled; skipped")
        continue
    res = validation.agreement(g.human_senior.astype(int), g.final.astype(int))
    res.update({"block": block, "model": MODEL})
    ads.write_json(REPO_DIR / "reports" / f"validation_{block}.json", res)
    rows.append(g.assign(block=block)[["block", "sheet_id", "ad_id", "human_senior", "final"]])
    print(block, json.dumps(res))
if rows:
    pd.concat(rows).rename(columns={"final": "rater_final"}).to_csv(REPO_DIR / "reports" / "validation_labels.csv", index=False)
    print("written: reports/validation_<block>.json, reports/validation_labels.csv")


## End of unit: commit

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "symbolic-ref", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", 'nb03: blind hand-labelling sheet (LinkedIn 2024 stratified sample and all Monster 2019 advertisements); agreement, kappa, precision and recall of the rater per block in reports/validation_<block>.json'], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)
